# CP4 — Análise de consumo de energia elétrica

Base: `house-power_TRATADO_CP4SERS.csv` (medições de uma residência).

Roteiro da atividade:
1. Carregar a amostra e simplificar os nomes dos atributos
2. Valor máximo de potência ativa
3. Limite de 75% do máximo e DataFrame com os registros acima dele
4. Quantidade e percentual dos registros selecionados
5. Corrente média da amostra
6. Segundo DataFrame com as duas condições ao mesmo tempo
7. Comparação dos dois conjuntos e interpretação

## 1. Carregar a amostra e simplificar os nomes

O arquivo tem duas linhas extras logo abaixo do cabeçalho: uma com o tipo de cada
atributo (`continuous`) e outra em branco. Elas são descartadas na leitura com
`skiprows=[1, 2]`, senão o pandas leria tudo como texto.

Em seguida os nomes originais (em inglês) são trocados por nomes mais simples.

In [ ]:
import pandas as pd

# O arquivo CSV deve estar na mesma pasta deste notebook.
CAMINHO = "house-power_TRATADO_CP4SERS.csv"

# skiprows=[1, 2] descarta a linha de tipos ("continuous") e a linha em branco.
df = pd.read_csv(CAMINHO, skiprows=[1, 2])

print("Nomes originais:", list(df.columns))

# Nome original -> nome simplificado
df = df.rename(columns={
    "Global_active_power": "potencia_ativa",       # kW
    "Global_reactive_power": "potencia_reativa",   # kW
    "Voltage": "tensao",                           # V
    "Global_intensity": "corrente",                # A
    "Sub_metering_1": "submedicao_1",              # Wh
    "Sub_metering_2": "submedicao_2",              # Wh
    "Sub_metering_3": "submedicao_3",              # Wh
})

print("Nomes simplificados:", list(df.columns))
print("Registros:", df.shape[0], "| Colunas:", df.shape[1])

df.head()

Nomes originais: ['Global_active_power', 'Global_reactive_power', 'Voltage', 'Global_intensity', 'Sub_metering_1', 'Sub_metering_2', 'Sub_metering_3']
Nomes simplificados: ['potencia_ativa', 'potencia_reativa', 'tensao', 'corrente', 'submedicao_1', 'submedicao_2', 'submedicao_3']
Registros: 204928 | Colunas: 7


,potencia_ativa,potencia_reativa,tensao,corrente,submedicao_1,submedicao_2,submedicao_3
0,1.502,0.074,240.17,6.4,0,0,18
1,0.374,0.264,245.50,1.8,0,2,0
2,0.620,0.300,239.85,3.0,0,1,1
3,0.280,0.200,235.72,1.4,0,0,0
4,1.372,0.054,243.95,5.6,0,0,18


Conferência rápida: todas as colunas devem ser numéricas e sem valores ausentes.
Se aparecer `object` em alguma coluna, é sinal de que as linhas extras não foram removidas.

In [ ]:
df.info()

print("\nValores ausentes por coluna:")
print(df.isna().sum())

<class 'pandas.DataFrame'>
RangeIndex: 204928 entries, 0 to 204927
Data columns (total 7 columns):
 #   Column            Non-Null Count   Dtype  
---  ------            --------------   -----  
 0   potencia_ativa    204928 non-null  float64
 1   potencia_reativa  204928 non-null  float64
 2   tensao            204928 non-null  float64
 3   corrente          204928 non-null  float64
 4   submedicao_1      204928 non-null  int64  
 5   submedicao_2      204928 non-null  int64  
 6   submedicao_3      204928 non-null  int64  
dtypes: float64(4), int64(3)
memory usage: 10.9 MB

Valores ausentes por coluna:
potencia_ativa      0
potencia_reativa    0
tensao              0
corrente            0
submedicao_1        0
submedicao_2        0
submedicao_3        0
dtype: int64


## 2. Valor máximo de potência ativa

`.max()` retorna o maior valor da coluna.

In [ ]:
potencia_maxima = df["potencia_ativa"].max()

print(f"Potência ativa máxima: {potencia_maxima:.3f} kW")

Potência ativa máxima: 10.536 kW


## 3. Limite de 75% do máximo e DataFrame com os registros acima dele

O limite é 0.75 * máximo. A filtragem usa uma máscara booleana:
df"potencia_ativa" > limite gera True/False para cada linha, e o df[...]
mantém apenas as linhas marcadas como True.

In [ ]:
limite_potencia = 0.75 * potencia_maxima

df_potencia = df[df["potencia_ativa"] > limite_potencia]

print(f"Limite (75% do máximo): {limite_potencia:.3f} kW")
print("Registros acima do limite:", len(df_potencia))

df_potencia.head()

Limite (75% do máximo): 7.902 kW
Registros acima do limite: 58


,potencia_ativa,potencia_reativa,tensao,corrente,submedicao_1,submedicao_2,submedicao_3
3132,8.540,0.238,236.23,36.0,80,35,17
3630,8.278,0.088,231.50,35.6,0,71,17
10339,8.026,0.066,236.45,33.8,37,73,18
11695,8.102,0.190,229.31,35.4,35,35,17
27983,10.290,0.302,230.90,44.6,35,66,17


## 4. Quantidade e percentual dos registros selecionados

O percentual é a quantidade selecionada dividida pelo total de registros da amostra.

In [ ]:
total_registros = len(df)
qtd_potencia = len(df_potencia)
pct_potencia = qtd_potencia / total_registros * 100

print("Total da amostra:", total_registros)
print("Registros acima de 75% do máximo:", qtd_potencia)
print(f"Percentual: {pct_potencia:.4f}%")

Total da amostra: 204928
Registros acima de 75% do máximo: 58
Percentual: 0.0283%


## 5. Corrente média da amostra

Média calculada sobre a amostra inteira, não apenas sobre o subconjunto filtrado.

In [ ]:
corrente_media = df["corrente"].mean()

print(f"Corrente média da amostra: {corrente_media:.3f} A")

Corrente média da amostra: 4.634 A


## 6. Segundo DataFrame: potência acima de 75% do máximo **e** corrente acima da média

As duas condições são combinadas com & (E lógico). Cada condição precisa ficar
entre parênteses.

In [ ]:
df_potencia_corrente = df[
    (df["potencia_ativa"] > limite_potencia) & (df["corrente"] > corrente_media)
]

qtd_ambas = len(df_potencia_corrente)
pct_ambas = qtd_ambas / total_registros * 100

print("Registros com as duas condições:", qtd_ambas)
print(f"Percentual: {pct_ambas:.4f}%")

df_potencia_corrente.head()

Registros com as duas condições: 58
Percentual: 0.0283%


,potencia_ativa,potencia_reativa,tensao,corrente,submedicao_1,submedicao_2,submedicao_3
3132,8.540,0.238,236.23,36.0,80,35,17
3630,8.278,0.088,231.50,35.6,0,71,17
10339,8.026,0.066,236.45,33.8,37,73,18
11695,8.102,0.190,229.31,35.4,35,35,17
27983,10.290,0.302,230.90,44.6,35,66,17


## 7. Comparação dos dois conjuntos

Primeiro a tabela comparativa, depois os números que explicam o resultado.

In [ ]:
comparativo = pd.DataFrame({
    "conjunto": ["1 - só potência", "2 - potência e corrente"],
    "registros": [qtd_potencia, qtd_ambas],
    "% da amostra": [round(pct_potencia, 4), round(pct_ambas, 4)],
    "potencia_min": [df_potencia["potencia_ativa"].min(),
                     df_potencia_corrente["potencia_ativa"].min()],
    "corrente_min": [df_potencia["corrente"].min(),
                     df_potencia_corrente["corrente"].min()],
    "corrente_media": [df_potencia["corrente"].mean(),
                       df_potencia_corrente["corrente"].mean()],
})

comparativo

,conjunto,registros,% da amostra,potencia_min,corrente_min,corrente_media
0,1 - só potência,58,0.0283,7.91,33.4,36.003448
1,2 - potência e corrente,58,0.0283,7.91,33.4,36.003448


In [ ]:
removidos = qtd_potencia - qtd_ambas

print("Registros removidos pela segunda condição:", removidos)
print(f"Menor corrente do conjunto 1: {df_potencia['corrente'].min():.1f} A")
print(f"Corrente média da amostra (limite da 2a condição): {corrente_media:.3f} A")
print(f"Correlação entre potência ativa e corrente: {df['potencia_ativa'].corr(df['corrente']):.4f}")

Registros removidos pela segunda condição: 0
Menor corrente do conjunto 1: 33.4 A
Corrente média da amostra (limite da 2a condição): 4.634 A
Correlação entre potência ativa e corrente: 0.9989


### Resposta interpretativa

A inclusão da corrente como segunda condição **não alterou o resultado**: os dois
conjuntos têm os mesmos 58 registros (0,0283% da amostra) e nenhum registro foi
removido.

O motivo é físico e aparece nos números:

- Potência ativa e corrente são quase perfeitamente correlacionadas (r ≈ 0,9989).
  Numa instalação com tensão praticamente constante (~240 V), a potência é
  proporcional à corrente, então uma variável praticamente determina a outra.
- Os dois limites têm severidade muito diferente. O limite de potência (7,902 kW)
  seleciona apenas o topo extremo da distribuição, enquanto a corrente média
  (4,634 A) é um limite baixo, a média é puxada para baixo pela maioria dos
  registros de consumo pequeno.
- Como consequência, a menor corrente dentro do conjunto 1 já é 33,4 A, cerca de
  7 vezes a média. Todo registro que passa no primeiro filtro passa
  automaticamente no segundo.

Ou seja, a segunda condição é **redundante** neste caso: ela restringe um conjunto
que já está inteiramente contido nela. Um filtro adicional só reduz a seleção
quando a variável usada é independente (ou fracamente relacionada) da primeira.
Se algum registro tivesse sido removido aqui, isso indicaria uma anomalia,
potência alta com corrente baixa, o que fugiria da relação física esperada e
sugeriria erro de medição.

Para que a corrente atuasse de fato como filtro, seria preciso usar um limite mais
exigente (por exemplo, um percentil alto da corrente) ou combinar com uma variável
não correlacionada com a potência ativa.

### Resumo dos resultados

| Item | Resultado |
|---|---|
| Registros na amostra | 204.928 |
| Potência ativa máxima | 10,536 kW |
| Limite (75% do máximo) | 7,902 kW |
| Registros acima do limite | 58 |
| Percentual da amostra | 0,0283% |
| Corrente média da amostra | 4,634 A |
| Registros com as duas condições | 58 (0,0283%) |
| Diferença entre os conjuntos | 0 registros |